### Gold - fact_entrega e fact_ocorrencia

Mantidas separadas de fact_pedido_item por possuírem granularidade
própria: fact_entrega corresponde a 1 linha por entrega (um pedido pode
estar associado a mais de uma, conforme identificado na Silver), e
fact_ocorrencia a 1 linha por ticket (um pedido pode gerar mais de um
ticket). A combinação dessas fatos com a fato de item, via join 1-para-N,
duplicaria linhas. O relacionamento entre as fatos é feito por order_id.

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F

df_entregas = spark.table(f"{schema_name}.silver_entregas")
df_ocorrencias = spark.table(f"{schema_name}.silver_ocorrencias")
df_cab = spark.table(f"{schema_name}.silver_pedidos_cabecalho").select(
    "order_id", "customer_code", "seller_id", "order_date", "status_order"
)

In [ ]:
df_fact_entrega = (
    df_entregas.join(df_cab, "order_id", "left")
    .select(
        "delivery_id", "order_id",
        F.col("customer_code").alias("customer_id"), "seller_id", "order_date",
        "carrier_name", "carrier_mode", "delivery_status",
        "shipped_at", "delivered_at", "tempo_entrega_dias",
        F.col("destino_uf"), F.col("destino_regiao"),
        F.col("destination_city").alias("destino_cidade"),
        F.col("cost").alias("custo_frete"),
    )
)

print("Entregas sem pedido correspondente:", df_fact_entrega.filter(F.col("customer_id").isNull()).count())

df_fact_entrega.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.fact_entrega")
print("fact_entrega:", df_fact_entrega.count())

In [ ]:
df_fact_ocorrencia = (
    df_ocorrencias.join(df_cab, "order_id", "left")
    .select(
        "ticket_id", "order_id",
        F.col("customer_code").alias("customer_id"), "seller_id",
        F.col("order_date").alias("order_date_pedido"),
        "created_at", "event_type", "severity",
        F.col("status").alias("status_ticket"),
        "canal_atendimento", "sla_horas",
        F.col("status_order").alias("status_pedido_associado"),
    )
)

print("Ocorrências sem pedido correspondente:", df_fact_ocorrencia.filter(F.col("customer_id").isNull()).count())

df_fact_ocorrencia.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{schema_name}.fact_ocorrencia")
print("fact_ocorrencia:", df_fact_ocorrencia.count())

In [ ]:
tabelas_gold = [
    "dim_cliente", "dim_vendedor", "dim_produto", "dim_canal", "dim_regiao", "dim_tempo",
    "fact_pedido_item", "fact_entrega", "fact_ocorrencia",
]
resumo = [(t, spark.table(f"{schema_name}.{t}").count()) for t in tabelas_gold]
display(spark.createDataFrame(resumo, ["tabela", "linhas"]))